# Lesson 5 — Integer Programming Basics

## Learning objectives

1. Recognize when integrality matters and write IP/MILP formulations.
2. Apply the canonical IP modeling tricks (big-M, indicator constraints, SOS, fixed-cost charges).
3. Understand totally unimodular (TUM) matrices and why some "IPs" are really LPs.
4. Distinguish *strong* from *weak* formulations.

## 1. The integrality leap

LP is polynomial-time {cite:p}`Khachiyan1979,Karmarkar1984` ({cite:t}`Khachiyan1979`'s ellipsoid was the first poly-time algorithm; {cite:t}`Karmarkar1984` was the first practical interior-point one). Adding $x \in \mathbb{Z}^n$ makes the problem NP-hard {cite:p}`Wolsey1998`. Two reasons:

1. **Combinatorial:** the feasible set is a finite (or countable) set of points.
2. **Non-convex:** the convex hull of feasible integer points (the **integer hull**) is a polytope distinct from the LP relaxation; finding it is the heart of cutting-plane methods (Lesson 13).

## 2. Big-M

Encode "if $z = 1$ then $a^\top x \le b$, else no constraint":

$$a^\top x \le b + M (1 - z)$$

with $M$ a constant chosen so the inequality is vacuous when $z = 0$. Tight $M$ is critical: too-loose $M$ gives a weak LP relaxation and slow B&B {cite:p}`Williams2013,Belotti2009`.

> **Rule:** always derive the *smallest valid* $M$ from variable bounds. Never pick $10^9$ "to be safe."

## 3. Fixed-cost charges

$$\text{cost} = \sum_i (f_i z_i + c_i x_i), \quad x_i \le U_i z_i$$

$z_i = 1$ if facility $i$ is opened ($U_i$ = capacity). Classic facility-location pattern.

## 4. SOS / piecewise linear

A *Special Ordered Set Type 2* (SOS2) is a set of variables of which at most two consecutive may be nonzero. Used to encode piecewise linear $f(x)$:

$$x = \sum_k \lambda_k \hat x_k, \; f(x) = \sum_k \lambda_k f(\hat x_k), \; \lambda \in \text{SOS2}, \sum \lambda_k = 1.$$

Many MILP solvers (and `discopt`) accept SOS as a special constraint type with a tailored branching rule {cite:p}`Conforti2014`.

## 5. Totally unimodular (TUM)

A matrix $A$ is TUM if every square submatrix has determinant $0, +1,$ or $-1$. **Theorem:** if $A$ is TUM and $b$ is integer, the LP $\min c^\top x$ s.t. $Ax \le b, x \ge 0$ has integer optimal vertices automatically.

Standard examples: bipartite matching, network flow, assignment, shortest path, max flow {cite:p}`Conforti2014,Wolsey1998`. If you can recognize TUM structure, you don't need integer variables — the LP gives the IP answer.

## 6. Strong vs weak formulations

Two formulations of the same IP can have very different LP relaxations. The one with smaller LP-relaxation feasible set is **stronger** — its B&B tree is smaller. Example: facility location.

- *Aggregated:* $\sum_i x_{ij} \le M z_j$ for each facility $j$.
- *Disaggregated:* $x_{ij} \le z_j$ for each $(i, j)$ pair.

Disaggregated is stronger (and has more constraints). The trade-off shows up everywhere; understanding it is the difference between models that solve in seconds vs hours {cite:p}`Williams2013,Wolsey1998`.

## References
{cite:p}`Wolsey1998,Williams2013,Conforti2014` are the canonical IP textbooks. {cite:p}`Belotti2009` discusses bound tightening in MINLP.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm
